# Week 7: Sampling & Estimation — Applied Statistics with Python (2026)

How can we learn about a population of millions by studying just a few hundred? This week we explore **sampling methods**, the remarkable **Central Limit Theorem**, and how to build **confidence intervals** that quantify our uncertainty. We also introduce the powerful **bootstrap** — a computer-intensive resampling technique that works even when traditional formulas fail.

| # | Topic | Key Concepts |
|---|-------|--------------|
| 1 | Population vs Sample | Parameters vs statistics, why we sample |
| 2 | Sampling Methods | SRS, stratified, systematic, cluster |
| 3 | Sampling Bias | Selection bias, survivorship bias, response bias |
| 4 | Sampling Distribution | Distribution of a statistic, standard error |
| 5 | Central Limit Theorem | CLT statement, simulation demo, conditions |
| 6 | Point Estimation | Estimator properties: unbiased, consistent, efficient |
| 7 | Confidence Intervals (Mean) | z-interval, t-interval, interpretation |
| 8 | CI for Proportions | Wald interval, Wilson interval |
| 9 | Sample Size Determination | Margin of error, required n |
| 10 | Bootstrap Method | Resampling concept, bootstrap distribution |
| 11 | Bootstrap Confidence Intervals | Percentile, BCa methods |
| 12 | Practical Example | Complete estimation pipeline |
| 13 | Summary | Key functions and concepts |
| 14 | Homework | Practice exercises |

### Import all necessary libraries for this week's analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set consistent style for all plots
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# Reproducibility
np.random.seed(42)

print('All libraries loaded successfully!')

---
## 1. Population vs Sample

In statistics, we distinguish between:

- **Population**: The entire group we want to study (often huge or infinite)
- **Sample**: A subset of the population that we actually observe

| Concept | Population | Sample |
|---------|-----------|--------|
| Size | N (often very large) | n (manageable) |
| Mean | μ (parameter) | x̄ (statistic) |
| Std Dev | σ (parameter) | s (statistic) |
| Proportion | p (parameter) | p̂ (statistic) |
| Nature | Fixed but unknown | Varies from sample to sample |

**Why sample?** Because measuring every individual in a population is often impossible, too expensive, or destructive (e.g., testing if a light bulb works until it burns out).

Let's create a simulated population of 100,000 adult heights and compare population parameters with sample statistics.

In [ ]:
# Simulate a population of 100,000 adult heights (cm)
population = np.random.normal(loc=170, scale=8, size=100_000)

# Take a random sample of 50
sample = np.random.choice(population, size=50, replace=False)

# Compare population parameters vs sample statistics
print('=== Population Parameters ===')
print(f'  N = {len(population):,}')
print(f'  μ = {population.mean():.2f} cm')
print(f'  σ = {population.std():.2f} cm')

print('\n=== Sample Statistics ===')
print(f'  n = {len(sample)}')
print(f'  x̄ = {sample.mean():.2f} cm')
print(f'  s = {sample.std(ddof=1):.2f} cm')  # ddof=1 for sample std

print(f'\nDifference in means: {abs(population.mean() - sample.mean()):.2f} cm')

Different samples give different estimates. Let's visualize how sample means vary across 20 different samples from the same population.

In [ ]:
# Draw 20 different samples and compare their means
n_samples = 20
sample_size = 50
sample_means = [np.random.choice(population, size=sample_size, replace=False).mean() 
                for _ in range(n_samples)]

fig, ax = plt.subplots(figsize=(10, 5))

# Plot each sample mean
ax.scatter(range(n_samples), sample_means, color='steelblue', s=80, zorder=3)
ax.axhline(y=population.mean(), color='red', linestyle='--', linewidth=2, 
           label=f'Population μ = {population.mean():.2f}')

# Connect dots to the population mean for visual comparison
for i, m in enumerate(sample_means):
    ax.vlines(i, min(m, population.mean()), max(m, population.mean()), 
              colors='gray', alpha=0.5, linewidth=1)

ax.set_xlabel('Sample Number', fontsize=12)
ax.set_ylabel('Sample Mean (cm)', fontsize=12)
ax.set_title('Variability of Sample Means (n=50 each)', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f'Range of sample means: {min(sample_means):.2f} to {max(sample_means):.2f}')

---
## 2. Sampling Methods

Not all samples are created equal. The **method** of sampling determines whether our conclusions are valid. Here are the main probability sampling methods:

| Method | Description | When to Use | Pros | Cons |
|--------|-------------|-------------|------|------|
| **Simple Random (SRS)** | Every unit has equal chance | Homogeneous population | Simple, unbiased | May miss subgroups |
| **Stratified** | Divide into strata, sample each | Known subgroups matter | Guarantees representation | Need strata info |
| **Systematic** | Every k-th unit | Ordered list available | Easy to implement | Periodic patterns risk |
| **Cluster** | Random groups, sample all within | Geographic spread | Cost-effective | Higher variance |

### 2.1 Simple Random Sampling (SRS)

Every member of the population has an equal probability of being selected. This is the gold standard of sampling.

Demonstrate simple random sampling using both NumPy and Pandas methods.

In [ ]:
# Create a population DataFrame (e.g., a school with 500 students)
np.random.seed(42)
school = pd.DataFrame({
    'student_id': range(1, 501),
    'grade': np.random.choice(['A', 'B', 'C', 'D'], size=500, p=[0.15, 0.35, 0.35, 0.15]),
    'score': np.random.normal(75, 12, 500).clip(0, 100).round(1)
})

# Method 1: NumPy random choice (returns array)
srs_indices = np.random.choice(school.index, size=30, replace=False)
srs_sample_np = school.loc[srs_indices]

# Method 2: Pandas .sample() — preferred for DataFrames
srs_sample_pd = school.sample(n=30, random_state=42)

print(f'Population mean score: {school["score"].mean():.2f}')
print(f'SRS sample mean (n=30): {srs_sample_pd["score"].mean():.2f}')
print(f'\nSample grade distribution:')
print(srs_sample_pd['grade'].value_counts(normalize=True).sort_index())

### 2.2 Stratified Sampling

We divide the population into homogeneous subgroups (**strata**) and sample from each stratum. This guarantees that important subgroups are represented proportionally.

Implement stratified sampling by grade, ensuring proportional representation from each grade level.

In [ ]:
def stratified_sample(df, strata_col, n, random_state=42):
    """Proportional stratified sampling."""
    # Calculate proportion of each stratum
    proportions = df[strata_col].value_counts(normalize=True)
    
    samples = []
    for stratum, prop in proportions.items():
        stratum_data = df[df[strata_col] == stratum]
        n_stratum = max(1, round(n * prop))  # At least 1 from each stratum
        samples.append(stratum_data.sample(n=n_stratum, random_state=random_state))
    
    return pd.concat(samples)

# Stratified sample: 30 students, proportional by grade
strat_sample = stratified_sample(school, 'grade', n=30)

print('Population grade distribution:')
print(school['grade'].value_counts(normalize=True).sort_index())
print(f'\nStratified sample grade distribution (n={len(strat_sample)}):')
print(strat_sample['grade'].value_counts(normalize=True).sort_index())
print(f'\nStratified sample mean score: {strat_sample["score"].mean():.2f}')

### 2.3 Systematic Sampling

Select every **k-th** element from an ordered list. The starting point is chosen randomly.

Implement systematic sampling by selecting every k-th student from the list.

In [ ]:
def systematic_sample(df, n, random_state=42):
    """Systematic sampling: every k-th element."""
    N = len(df)
    k = N // n  # Sampling interval
    
    rng = np.random.RandomState(random_state)
    start = rng.randint(0, k)  # Random starting point
    
    indices = list(range(start, N, k))[:n]  # Select every k-th element
    return df.iloc[indices]

sys_sample = systematic_sample(school, n=30)

print(f'Sampling interval k = {len(school) // 30}')
print(f'Systematic sample mean score: {sys_sample["score"].mean():.2f}')
print(f'Sample size: {len(sys_sample)}')
print(f'\nFirst 5 selected indices: {list(sys_sample.index[:5])}')

### 2.4 Cluster Sampling

Divide the population into **clusters** (natural groups), randomly select some clusters, then include **all** members of selected clusters. Useful when the population is geographically spread out.

Simulate cluster sampling by randomly selecting classrooms and including all students from those classrooms.

In [ ]:
# Add classroom (cluster) assignment: 20 classrooms of ~25 students each
school['classroom'] = np.random.choice(range(1, 21), size=500)

# Cluster sampling: randomly select 3 classrooms
np.random.seed(42)
selected_clusters = np.random.choice(school['classroom'].unique(), size=3, replace=False)
cluster_sample = school[school['classroom'].isin(selected_clusters)]

print(f'Selected classrooms: {sorted(selected_clusters)}')
print(f'Cluster sample size: {len(cluster_sample)}')
print(f'Cluster sample mean score: {cluster_sample["score"].mean():.2f}')
print(f'Population mean score: {school["score"].mean():.2f}')

### 2.5 Comparing Sampling Methods

Let's compare the accuracy of different sampling methods by repeating each method 1000 times and examining the distribution of sample means.

Run a simulation to compare the variability of sample means across the four sampling methods.

In [ ]:
n_reps = 1000
n_sample = 30
pop_mean = school['score'].mean()

# Collect means from each method
srs_means = [school.sample(n=n_sample, random_state=i)['score'].mean() for i in range(n_reps)]
strat_means = [stratified_sample(school, 'grade', n=n_sample, random_state=i)['score'].mean() 
               for i in range(n_reps)]
sys_means = [systematic_sample(school, n=n_sample, random_state=i)['score'].mean() 
             for i in range(n_reps)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

methods = ['SRS', 'Stratified', 'Systematic']
all_means = [srs_means, strat_means, sys_means]
colors = ['steelblue', 'coral', 'seagreen']

for ax, means, method, color in zip(axes, all_means, methods, colors):
    ax.hist(means, bins=30, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(pop_mean, color='red', linestyle='--', linewidth=2, label=f'μ = {pop_mean:.2f}')
    ax.set_title(f'{method}\nSD = {np.std(means):.3f}', fontsize=13)
    ax.set_xlabel('Sample Mean Score', fontsize=11)
    ax.legend(fontsize=10)

axes[0].set_ylabel('Frequency', fontsize=11)
fig.suptitle('Comparison of Sampling Methods (1000 repetitions, n=30)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Sampling Bias

**Sampling bias** occurs when some members of the population are systematically more likely to be selected than others. Even large samples cannot fix a biased sampling method.

| Bias Type | Description | Example |
|-----------|-------------|---------|
| **Selection bias** | Sample not representative | Surveying only daytime shoppers |
| **Survivorship bias** | Only "survivors" observed | Studying successful companies only |
| **Response bias** | Non-response differs from response | Satisfied customers more likely to respond |
| **Convenience bias** | Sample based on easy access | Polling friends about political views |
| **Voluntary response** | Self-selected participants | Online review ratings (extreme views) |

Demonstrate how a biased sample systematically misestimates the population parameter, no matter the sample size.

In [ ]:
# Simulate: population income (right-skewed)
np.random.seed(42)
incomes = np.random.lognormal(mean=10.5, sigma=0.6, size=50_000)  # ~$40k median

# Biased sample: wealthier people more likely to respond to a survey
# Selection probability proportional to income
probs = incomes / incomes.sum()  # Higher income → higher selection chance
biased_sample = np.random.choice(incomes, size=500, replace=False, p=probs)

# Unbiased (SRS) sample
unbiased_sample = np.random.choice(incomes, size=500, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: compare distributions
axes[0].hist(incomes, bins=80, density=True, alpha=0.4, color='gray', label='Population')
axes[0].hist(unbiased_sample, bins=40, density=True, alpha=0.6, color='steelblue', label='Unbiased (SRS)')
axes[0].hist(biased_sample, bins=40, density=True, alpha=0.6, color='coral', label='Biased')
axes[0].set_xlabel('Income ($)', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Biased vs Unbiased Sampling', fontsize=13)
axes[0].set_xlim(0, 200_000)
axes[0].legend(fontsize=10)

# Right: means comparison
means = [np.mean(incomes), np.mean(unbiased_sample), np.mean(biased_sample)]
labels = ['Population\nμ', 'Unbiased\nSample', 'Biased\nSample']
bars = axes[1].bar(labels, means, color=['gray', 'steelblue', 'coral'], alpha=0.8)
for bar, m in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500, 
                 f'${m:,.0f}', ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Mean Income ($)', fontsize=12)
axes[1].set_title('Effect of Sampling Bias on Estimates', fontsize=13)

plt.tight_layout()
plt.show()

---
## 4. Sampling Distribution

The **sampling distribution** is the distribution of a statistic (like the mean) computed from many repeated samples of the same size from the same population.

Key concepts:
- The sampling distribution of x̄ is centered at μ (the population mean)
- Its spread is called the **standard error (SE)**: SE = σ / √n
- Larger samples → smaller SE → more precise estimates

This is NOT the distribution of the data itself — it's the distribution of the *statistic* across repeated samples.

Build a sampling distribution by drawing 2000 samples of size n=30 from our height population and plotting the distribution of sample means.

In [ ]:
# Build sampling distribution of the mean
n = 30
n_reps = 2000
sampling_means = [np.random.choice(population, size=n, replace=True).mean() 
                  for _ in range(n_reps)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: population distribution
axes[0].hist(population, bins=80, color='lightcoral', alpha=0.7, edgecolor='white', density=True)
axes[0].set_title(f'Population Distribution\nμ = {population.mean():.2f}, σ = {population.std():.2f}', fontsize=13)
axes[0].set_xlabel('Height (cm)', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)

# Right: sampling distribution of x̄
axes[1].hist(sampling_means, bins=50, color='steelblue', alpha=0.7, edgecolor='white', density=True)
axes[1].set_title(f'Sampling Distribution of x̄ (n={n})\n'
                   f'Mean = {np.mean(sampling_means):.2f}, SE = {np.std(sampling_means):.2f}', fontsize=13)
axes[1].set_xlabel('Sample Mean (cm)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)

# Theoretical SE
theoretical_se = population.std() / np.sqrt(n)
print(f'Theoretical SE = σ/√n = {population.std():.2f}/√{n} = {theoretical_se:.2f}')
print(f'Observed SE (from simulation) = {np.std(sampling_means):.2f}')

plt.tight_layout()
plt.show()

Show how standard error decreases as sample size increases — demonstrating the √n relationship.

In [ ]:
# How SE changes with sample size
sample_sizes = [5, 10, 20, 30, 50, 100, 200, 500]
observed_se = []
theoretical_se_list = []

for n in sample_sizes:
    means = [np.random.choice(population, size=n, replace=True).mean() for _ in range(2000)]
    observed_se.append(np.std(means))
    theoretical_se_list.append(population.std() / np.sqrt(n))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sample_sizes, theoretical_se_list, 'r--o', linewidth=2, markersize=8, label='Theoretical: σ/√n')
ax.plot(sample_sizes, observed_se, 'b-s', linewidth=2, markersize=8, label='Observed SE (simulation)')
ax.set_xlabel('Sample Size (n)', fontsize=12)
ax.set_ylabel('Standard Error', fontsize=12)
ax.set_title('Standard Error Decreases with Sample Size', fontsize=14)
ax.legend(fontsize=12)
ax.set_xscale('log')  # Log scale to see the relationship clearly
plt.tight_layout()
plt.show()

---
## 5. Central Limit Theorem (CLT)

The **Central Limit Theorem** is one of the most important results in all of statistics:

> If we draw independent samples of size n from **any** population with mean μ and finite variance σ², then as n increases, the sampling distribution of x̄ approaches a **normal distribution**:
>
> x̄ ~ N(μ, σ²/n)

**Key points:**
- Works for **any** population shape (uniform, skewed, bimodal, etc.)
- Generally, n ≥ 30 is sufficient for most populations
- More skewed populations need larger n
- This is *why* the normal distribution appears so often in statistics

### 5.1 CLT Simulation: From Any Distribution to Normal

Let's demonstrate the CLT with four very different population distributions and watch their sampling distributions become normal.

Simulate sampling distributions from four non-normal populations (uniform, exponential, bimodal, heavily skewed) at different sample sizes to visualize the CLT.

In [ ]:
np.random.seed(42)

# Four different population distributions
populations = {
    'Uniform': np.random.uniform(0, 10, 100_000),
    'Exponential': np.random.exponential(2, 100_000),
    'Bimodal': np.concatenate([np.random.normal(3, 0.8, 50_000), 
                               np.random.normal(7, 0.8, 50_000)]),
    'Chi-squared (df=2)': np.random.chisquare(2, 100_000)
}

sample_sizes_clt = [1, 5, 30, 100]

fig, axes = plt.subplots(4, 4, figsize=(16, 14))

for row, (name, pop) in enumerate(populations.items()):
    for col, n in enumerate(sample_sizes_clt):
        # Draw 3000 samples of size n, compute mean of each
        means = [np.random.choice(pop, size=n, replace=True).mean() for _ in range(3000)]
        
        ax = axes[row][col]
        ax.hist(means, bins=40, density=True, color='steelblue', alpha=0.7, edgecolor='white')
        
        # Overlay normal curve for comparison
        if n > 1:
            x = np.linspace(min(means), max(means), 100)
            ax.plot(x, stats.norm.pdf(x, np.mean(means), np.std(means)), 
                    'r-', linewidth=2)
        
        if row == 0:
            ax.set_title(f'n = {n}', fontsize=13, fontweight='bold')
        if col == 0:
            ax.set_ylabel(name, fontsize=12, fontweight='bold')

fig.suptitle('Central Limit Theorem: Sampling Distributions Become Normal', 
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Interactive CLT: Step-by-Step Animation

Let's watch the sampling distribution build up as we draw more and more samples from an exponential distribution.

Show how the sampling distribution evolves as we accumulate more samples — from just 10 samples to 2000.

In [ ]:
# CLT step-by-step: exponential population
np.random.seed(42)
exp_pop = np.random.exponential(scale=5, size=100_000)  # Mean=5, heavily right-skewed
n = 30

# Accumulate sample means
all_means = [np.random.choice(exp_pop, size=n, replace=True).mean() for _ in range(2000)]

steps = [10, 50, 200, 2000]
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

for ax, step in zip(axes, steps):
    ax.hist(all_means[:step], bins=30, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    
    # Overlay theoretical normal
    x = np.linspace(2, 8, 100)
    ax.plot(x, stats.norm.pdf(x, exp_pop.mean(), exp_pop.std() / np.sqrt(n)), 
            'r-', linewidth=2, label='Theoretical Normal')
    ax.set_title(f'{step} samples', fontsize=13)
    ax.set_xlabel('Sample Mean', fontsize=11)
    ax.axvline(exp_pop.mean(), color='red', linestyle='--', alpha=0.5)

axes[0].set_ylabel('Density', fontsize=11)
axes[-1].legend(fontsize=10)
fig.suptitle(f'Building the Sampling Distribution (Exponential Pop, n={n})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5.3 CLT Conditions and Limitations

The CLT requires:
1. **Independence**: Samples must be independent (usually satisfied by random sampling)
2. **Finite variance**: The population must have finite variance
3. **Sample size**: n ≥ 30 is a rule of thumb, but:
   - Symmetric distributions: even n = 10 works well
   - Highly skewed: may need n ≥ 50 or more
   - Heavy-tailed (e.g., Cauchy): CLT may not apply!

Demonstrate a case where the CLT fails — the Cauchy distribution has no finite mean or variance.

In [ ]:
# CLT failure case: Cauchy distribution (infinite variance)
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, n in zip(axes, [5, 30, 200]):
    # Cauchy distribution: heavy tails, no mean/variance
    cauchy_means = [np.random.standard_cauchy(n).mean() for _ in range(3000)]
    
    # Clip extreme values for visualization
    clipped = np.clip(cauchy_means, -20, 20)
    ax.hist(clipped, bins=60, density=True, color='darkorange', alpha=0.7, edgecolor='white')
    ax.set_title(f'Cauchy: n = {n}\n(Still not normal!)', fontsize=13)
    ax.set_xlabel('Sample Mean', fontsize=11)
    ax.set_xlim(-20, 20)

axes[0].set_ylabel('Density', fontsize=11)
fig.suptitle('CLT FAILS for Cauchy Distribution (Infinite Variance)', 
             fontsize=14, fontweight='bold', color='red', y=1.02)
plt.tight_layout()
plt.show()
print('The Cauchy distribution has no finite mean or variance.')
print('No matter how large n is, the sampling distribution of the mean does NOT converge to normal!')

---
## 6. Point Estimation

A **point estimator** is a statistic that provides a single "best guess" for a population parameter.

| Parameter | Estimator | Properties |
|-----------|-----------|------------|
| μ (mean) | x̄ = Σxᵢ/n | Unbiased, consistent, efficient |
| σ² (variance) | s² = Σ(xᵢ-x̄)²/(n-1) | Unbiased (with n-1) |
| p (proportion) | p̂ = X/n | Unbiased, consistent |

**Desirable properties of estimators:**
- **Unbiased**: E[θ̂] = θ (correct on average)
- **Consistent**: θ̂ → θ as n → ∞ (gets better with more data)
- **Efficient**: Smallest variance among unbiased estimators

Demonstrate unbiasedness: show that the average of many x̄ values converges to the true population mean μ.

In [ ]:
# Demonstrate unbiasedness and consistency
np.random.seed(42)
true_mean = population.mean()

# Unbiasedness: average of many sample means ≈ μ
sample_means_demo = [np.random.choice(population, size=30).mean() for _ in range(5000)]
print(f'True population mean (μ): {true_mean:.4f}')
print(f'Average of 5000 sample means: {np.mean(sample_means_demo):.4f}')
print(f'Difference: {abs(true_mean - np.mean(sample_means_demo)):.4f}')

# Consistency: larger samples → less variation
print('\n=== Consistency: SE decreases with n ===')
for n in [10, 30, 100, 500, 1000]:
    means = [np.random.choice(population, size=n).mean() for _ in range(1000)]
    print(f'  n = {n:>4}: SE = {np.std(means):.4f}')

Show why we divide by (n-1) instead of n when computing sample variance — demonstrating Bessel's correction.

In [ ]:
# Why n-1? Bessel's correction for unbiased variance
np.random.seed(42)
true_var = population.var()  # Population variance (divides by N)

n = 10
n_reps = 10_000

biased_vars = []    # Using n in denominator
unbiased_vars = []  # Using n-1 in denominator (Bessel's correction)

for _ in range(n_reps):
    s = np.random.choice(population, size=n)
    biased_vars.append(np.var(s, ddof=0))    # ddof=0 → divide by n
    unbiased_vars.append(np.var(s, ddof=1))   # ddof=1 → divide by (n-1)

print(f'True population variance: {true_var:.4f}')
print(f'Average biased variance (÷n):   {np.mean(biased_vars):.4f}  ← systematically too low!')
print(f'Average unbiased variance (÷n-1): {np.mean(unbiased_vars):.4f}  ← correct on average')

---
## 7. Confidence Intervals for the Mean

A **confidence interval (CI)** provides a range of plausible values for a population parameter, along with a confidence level (typically 95%).

**Interpretation:** A 95% CI means: if we repeated the sampling process many times and built a CI each time, about 95% of those intervals would contain the true parameter.

**It does NOT mean:** "There is a 95% probability that μ is in this interval" (μ is fixed, not random!).

### Formula:

| Scenario | Formula | Use When |
|----------|---------|----------|
| σ known | x̄ ± z* · σ/√n | Rarely in practice |
| σ unknown | x̄ ± t* · s/√n | Almost always |

Where z* and t* are critical values for the desired confidence level.

### 7.1 Z-Interval (σ Known)

When the population standard deviation σ is known (rare in practice), we use the z-distribution.

Compute a 95% z-interval for the mean height, assuming σ is known from the population.

In [ ]:
# Z-interval: when σ is known
np.random.seed(42)
sample_data = np.random.choice(population, size=50, replace=False)
sigma = population.std()  # Known population σ

x_bar = sample_data.mean()
n = len(sample_data)
confidence = 0.95

# Critical value z*
z_star = stats.norm.ppf((1 + confidence) / 2)  # 1.96 for 95%
margin_of_error = z_star * sigma / np.sqrt(n)

ci_lower = x_bar - margin_of_error
ci_upper = x_bar + margin_of_error

print(f'Sample mean (x̄): {x_bar:.2f}')
print(f'Known σ: {sigma:.2f}')
print(f'z* for 95% CI: {z_star:.4f}')
print(f'Margin of error: {margin_of_error:.2f}')
print(f'\n95% Z-Confidence Interval: ({ci_lower:.2f}, {ci_upper:.2f})')
print(f'True population mean: {population.mean():.2f}')
print(f'Does CI contain μ? {ci_lower < population.mean() < ci_upper}')

### 7.2 T-Interval (σ Unknown — The Common Case)

In practice, we almost never know σ. We estimate it with the sample standard deviation s, and use the **t-distribution** with (n-1) degrees of freedom. The t-distribution has heavier tails than the normal, accounting for the extra uncertainty.

Compute a 95% t-interval for the mean height using the sample standard deviation s.

In [ ]:
# T-interval: when σ is unknown (the common case)
s = sample_data.std(ddof=1)  # Sample std with Bessel's correction
df = n - 1                    # Degrees of freedom

# Critical value t*
t_star = stats.t.ppf((1 + confidence) / 2, df=df)
me_t = t_star * s / np.sqrt(n)

ci_lower_t = x_bar - me_t
ci_upper_t = x_bar + me_t

print(f'Sample mean (x̄): {x_bar:.2f}')
print(f'Sample std (s): {s:.2f}')
print(f't* for 95% CI (df={df}): {t_star:.4f}')
print(f'Margin of error: {me_t:.2f}')
print(f'\n95% T-Confidence Interval: ({ci_lower_t:.2f}, {ci_upper_t:.2f})')

# Shortcut: scipy.stats.t.interval()
ci_scipy = stats.t.interval(confidence, df=df, loc=x_bar, scale=s/np.sqrt(n))
print(f'\nUsing scipy shortcut: ({ci_scipy[0]:.2f}, {ci_scipy[1]:.2f})')

### 7.3 Visualizing Confidence Level

Let's build 100 confidence intervals and see how many actually contain the true population mean.

Draw 100 samples, build a 95% CI for each, and visualize which intervals capture the true mean — demonstrating the frequentist interpretation.

In [ ]:
# 100 confidence intervals — how many contain μ?
np.random.seed(42)
n_intervals = 100
n = 30
mu = population.mean()

fig, ax = plt.subplots(figsize=(12, 10))

contains_mu = 0
for i in range(n_intervals):
    s = np.random.choice(population, size=n, replace=False)
    x_bar = s.mean()
    se = s.std(ddof=1) / np.sqrt(n)
    ci = stats.t.interval(0.95, df=n-1, loc=x_bar, scale=se)
    
    # Check if CI contains true mean
    if ci[0] <= mu <= ci[1]:
        color = 'steelblue'
        contains_mu += 1
    else:
        color = 'red'  # Missed!
    
    ax.plot([ci[0], ci[1]], [i, i], color=color, linewidth=1.5, alpha=0.7)
    ax.plot(x_bar, i, 'o', color=color, markersize=3)

ax.axvline(mu, color='darkred', linestyle='--', linewidth=2, label=f'True μ = {mu:.2f}')
ax.set_xlabel('Height (cm)', fontsize=12)
ax.set_ylabel('Sample Number', fontsize=12)
ax.set_title(f'100 Confidence Intervals (95% level)\n'
             f'{contains_mu}/100 contain the true mean (red = missed)', fontsize=14)
ax.legend(fontsize=12, loc='upper right')
plt.tight_layout()
plt.show()

print(f'{contains_mu}% of intervals captured the true mean μ')

### 7.4 Effect of Confidence Level and Sample Size

Higher confidence → wider interval. Larger sample → narrower interval.

Compare how CI width changes with confidence level (90%, 95%, 99%) and with sample size.

In [ ]:
np.random.seed(42)
sample_demo = np.random.choice(population, size=50)

# Effect of confidence level
print('=== Effect of Confidence Level (n=50) ===')
for conf in [0.90, 0.95, 0.99]:
    ci = stats.t.interval(conf, df=49, loc=sample_demo.mean(), 
                          scale=sample_demo.std(ddof=1)/np.sqrt(50))
    width = ci[1] - ci[0]
    print(f'  {conf*100:.0f}% CI: ({ci[0]:.2f}, {ci[1]:.2f})  width = {width:.2f}')

# Effect of sample size
print('\n=== Effect of Sample Size (95% CI) ===')
for n in [10, 30, 50, 100, 500]:
    s = np.random.choice(population, size=n, replace=False)
    ci = stats.t.interval(0.95, df=n-1, loc=s.mean(), 
                          scale=s.std(ddof=1)/np.sqrt(n))
    width = ci[1] - ci[0]
    print(f'  n = {n:>3}: ({ci[0]:.2f}, {ci[1]:.2f})  width = {width:.2f}')

---
## 8. Confidence Interval for Proportions

When estimating a population proportion p (e.g., approval rate, defect rate), we use different formulas.

| Method | Formula | Notes |
|--------|---------|-------|
| **Wald** | p̂ ± z* · √(p̂(1-p̂)/n) | Simple but unreliable for small n or extreme p |
| **Wilson** | More complex, better coverage | Recommended, used by `statsmodels` |

The Wald interval is the textbook formula but can give poor coverage (below 95%) when p is near 0 or 1, or when n is small.

Compute both Wald and Wilson confidence intervals for a proportion, and compare them.

In [ ]:
from statsmodels.stats.proportion import proportion_confint

# Example: 120 out of 500 customers purchased a product
successes = 120
n = 500
p_hat = successes / n  # Sample proportion

# Method 1: Wald interval (manual)
z_star = stats.norm.ppf(0.975)  # 1.96
se_wald = np.sqrt(p_hat * (1 - p_hat) / n)
wald_ci = (p_hat - z_star * se_wald, p_hat + z_star * se_wald)

# Method 2: Wald interval (statsmodels)
wald_ci_sm = proportion_confint(successes, n, alpha=0.05, method='normal')

# Method 3: Wilson interval (recommended)
wilson_ci = proportion_confint(successes, n, alpha=0.05, method='wilson')

print(f'Sample proportion p̂ = {p_hat:.4f}')
print(f'\nWald CI (manual):      ({wald_ci[0]:.4f}, {wald_ci[1]:.4f})')
print(f'Wald CI (statsmodels): ({wald_ci_sm[0]:.4f}, {wald_ci_sm[1]:.4f})')
print(f'Wilson CI:             ({wilson_ci[0]:.4f}, {wilson_ci[1]:.4f})')

Show why the Wilson interval is preferred — simulate coverage rates for the Wald vs Wilson interval when the true proportion is extreme.

In [ ]:
# Simulation: compare Wald vs Wilson coverage for extreme proportions
np.random.seed(42)
true_p_values = [0.05, 0.10, 0.30, 0.50]  # Different true proportions
n = 40  # Small sample size
n_sim = 5000

print(f'Coverage rates for n={n}, 5000 simulations each:')
print(f'{"True p":>8} {"Wald":>10} {"Wilson":>10}')
print('-' * 30)

for true_p in true_p_values:
    wald_cover = 0
    wilson_cover = 0
    
    for _ in range(n_sim):
        x = np.random.binomial(n, true_p)  # Simulate count
        
        # Wald CI
        wald = proportion_confint(x, n, alpha=0.05, method='normal')
        if wald[0] <= true_p <= wald[1]:
            wald_cover += 1
        
        # Wilson CI
        wilson = proportion_confint(x, n, alpha=0.05, method='wilson')
        if wilson[0] <= true_p <= wilson[1]:
            wilson_cover += 1
    
    print(f'{true_p:>8.2f} {wald_cover/n_sim:>9.1%} {wilson_cover/n_sim:>9.1%}')

print('\n→ Wilson consistently achieves closer to 95% coverage, especially for extreme p!')

---
## 9. Sample Size Determination

Before collecting data, we often need to decide **how many observations** to collect. The required sample size depends on:

1. **Desired margin of error (E)**: How precise do we need?
2. **Confidence level**: Higher confidence → larger n
3. **Population variability (σ or p)**: More variable → larger n

### Formulas:

| For | Formula | Variables |
|-----|---------|----------|
| Mean | n = (z* · σ / E)² | E = desired margin of error |
| Proportion | n = z*² · p(1-p) / E² | Use p=0.5 if unknown (conservative) |

Calculate the required sample size for estimating a population mean and a population proportion with a specified margin of error.

In [ ]:
def sample_size_mean(sigma, E, confidence=0.95):
    """Required n for estimating a mean with margin of error E."""
    z_star = stats.norm.ppf((1 + confidence) / 2)
    n = (z_star * sigma / E) ** 2
    return int(np.ceil(n))  # Always round up

def sample_size_proportion(p_est, E, confidence=0.95):
    """Required n for estimating a proportion with margin of error E."""
    z_star = stats.norm.ppf((1 + confidence) / 2)
    n = z_star**2 * p_est * (1 - p_est) / E**2
    return int(np.ceil(n))

# Example 1: Estimate average height within ±1 cm (σ ≈ 8 cm)
n_needed = sample_size_mean(sigma=8, E=1, confidence=0.95)
print(f'Example 1: Estimate mean height within ±1 cm')
print(f'  Required n = {n_needed}')

# Example 2: Estimate election support within ±3% (p unknown → use 0.5)
n_needed2 = sample_size_proportion(p_est=0.5, E=0.03, confidence=0.95)
print(f'\nExample 2: Estimate proportion within ±3%')
print(f'  Required n = {n_needed2}')

# How n changes with desired precision
print('\n=== Required n for different margins of error (mean, σ=8) ===')
for E in [0.5, 1.0, 2.0, 3.0, 5.0]:
    print(f'  E = ±{E:.1f} cm → n = {sample_size_mean(8, E)}')

Visualize how required sample size explodes as the desired margin of error decreases — the cost of precision.

In [ ]:
# Visualize: n vs margin of error
E_values = np.linspace(0.3, 5, 100)
n_values_90 = [(stats.norm.ppf(0.95) * 8 / E) ** 2 for E in E_values]
n_values_95 = [(stats.norm.ppf(0.975) * 8 / E) ** 2 for E in E_values]
n_values_99 = [(stats.norm.ppf(0.995) * 8 / E) ** 2 for E in E_values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(E_values, n_values_90, 'g-', linewidth=2, label='90% confidence')
ax.plot(E_values, n_values_95, 'b-', linewidth=2, label='95% confidence')
ax.plot(E_values, n_values_99, 'r-', linewidth=2, label='99% confidence')

ax.set_xlabel('Margin of Error (cm)', fontsize=12)
ax.set_ylabel('Required Sample Size (n)', fontsize=12)
ax.set_title('Sample Size vs. Desired Margin of Error (σ = 8)', fontsize=14)
ax.legend(fontsize=12)
ax.set_ylim(0, 5000)
ax.axhline(y=1000, color='gray', linestyle=':', alpha=0.5)
ax.text(4.5, 1050, 'n=1000', fontsize=10, color='gray')
plt.tight_layout()
plt.show()

---
## 10. The Bootstrap Method

The **bootstrap** is a powerful resampling technique introduced by Bradley Efron (1979). Instead of relying on formulas and distributional assumptions, we:

1. Treat our sample as a "stand-in" for the population
2. **Resample with replacement** from our sample (same size n)
3. Compute the statistic of interest for each resample
4. Repeat thousands of times to build a **bootstrap distribution**

**Why bootstrap?**
- No distributional assumptions needed
- Works for complex statistics (median, trimmed mean, correlation, etc.)
- Can estimate standard errors when formulas don't exist

**Key idea:** The variability of the statistic across bootstrap resamples approximates the variability across real repeated samples.

### 10.1 Bootstrap Basics: Resampling with Replacement

Let's start with a simple example to understand how bootstrap resampling works.

Demonstrate the bootstrap procedure step by step: resample with replacement from the original sample, compute the statistic each time.

In [ ]:
# Simple bootstrap example
np.random.seed(42)

# Original sample: 15 exam scores
original_sample = np.array([72, 85, 91, 68, 77, 83, 95, 62, 88, 79, 74, 90, 81, 67, 86])
print(f'Original sample (n={len(original_sample)}): {original_sample}')
print(f'Original sample mean: {original_sample.mean():.2f}')
print(f'Original sample median: {np.median(original_sample):.2f}')

# One bootstrap resample: sample WITH replacement, same size n
boot_resample = np.random.choice(original_sample, size=len(original_sample), replace=True)
print(f'\nBootstrap resample 1: {boot_resample}')
print(f'  Note: some values appear multiple times, some are missing')
print(f'  Bootstrap mean: {boot_resample.mean():.2f}')

# Show a few more resamples
print('\nFive bootstrap means:')
for i in range(5):
    boot = np.random.choice(original_sample, size=len(original_sample), replace=True)
    print(f'  Resample {i+1}: mean = {boot.mean():.2f}')

### 10.2 Bootstrap Distribution

Now let's generate thousands of bootstrap resamples and examine the distribution of the bootstrap statistic.

Create bootstrap distributions for both the mean and median, showing that bootstrap works for any statistic.

In [ ]:
n_boot = 10_000  # Number of bootstrap resamples

# Bootstrap distribution for the MEAN
boot_means = [np.random.choice(original_sample, size=len(original_sample), 
                                replace=True).mean() 
              for _ in range(n_boot)]

# Bootstrap distribution for the MEDIAN
boot_medians = [np.median(np.random.choice(original_sample, size=len(original_sample), 
                                            replace=True)) 
                for _ in range(n_boot)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bootstrap distribution of mean
axes[0].hist(boot_means, bins=50, color='steelblue', alpha=0.7, edgecolor='white', density=True)
axes[0].axvline(original_sample.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Sample mean = {original_sample.mean():.2f}')
axes[0].set_title(f'Bootstrap Distribution of Mean\nSE = {np.std(boot_means):.2f}', fontsize=13)
axes[0].set_xlabel('Mean', fontsize=12)
axes[0].legend(fontsize=10)

# Bootstrap distribution of median
axes[1].hist(boot_medians, bins=50, color='coral', alpha=0.7, edgecolor='white', density=True)
axes[1].axvline(np.median(original_sample), color='red', linestyle='--', linewidth=2,
                label=f'Sample median = {np.median(original_sample):.2f}')
axes[1].set_title(f'Bootstrap Distribution of Median\nSE = {np.std(boot_medians):.2f}', fontsize=13)
axes[1].set_xlabel('Median', fontsize=12)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'Bootstrap SE for mean:   {np.std(boot_means):.3f}')
print(f'Bootstrap SE for median: {np.std(boot_medians):.3f}')
print(f'\nFormula SE for mean:    {original_sample.std(ddof=1)/np.sqrt(len(original_sample)):.3f}')
print('(No simple formula exists for the SE of the median — bootstrap solves this!)')

---
## 11. Bootstrap Confidence Intervals

The bootstrap distribution can be used to construct confidence intervals in several ways:

| Method | Approach | Use When |
|--------|----------|----------|
| **Percentile** | Use quantiles of bootstrap distribution | Simple, first choice |
| **Basic (Reverse Percentile)** | 2θ̂ - quantiles | Symmetric bias correction |
| **BCa** | Bias-corrected and accelerated | Most accurate, handles skew |
| **Bootstrap-t** | Uses bootstrap SE estimates | When SE is hard to compute |

### 11.1 Percentile Method

The simplest bootstrap CI: take the α/2 and (1-α/2) percentiles of the bootstrap distribution.

Compute bootstrap percentile confidence intervals for both the mean and median.

In [ ]:
def bootstrap_ci_percentile(data, stat_func, n_boot=10_000, confidence=0.95, random_state=42):
    """Bootstrap percentile confidence interval."""
    np.random.seed(random_state)
    n = len(data)
    boot_stats = [stat_func(np.random.choice(data, size=n, replace=True)) 
                  for _ in range(n_boot)]
    
    alpha = 1 - confidence
    lower = np.percentile(boot_stats, 100 * alpha / 2)
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return lower, upper, boot_stats

# 95% Bootstrap CI for the mean
ci_mean_lower, ci_mean_upper, boot_m = bootstrap_ci_percentile(
    original_sample, np.mean)
print(f'95% Bootstrap CI for mean:   ({ci_mean_lower:.2f}, {ci_mean_upper:.2f})')

# 95% Bootstrap CI for the median
ci_med_lower, ci_med_upper, boot_med = bootstrap_ci_percentile(
    original_sample, np.median)
print(f'95% Bootstrap CI for median: ({ci_med_lower:.2f}, {ci_med_upper:.2f})')

# Compare with t-interval for the mean
t_ci = stats.t.interval(0.95, df=len(original_sample)-1, 
                        loc=original_sample.mean(),
                        scale=original_sample.std(ddof=1)/np.sqrt(len(original_sample)))
print(f'\n95% T-interval for mean:     ({t_ci[0]:.2f}, {t_ci[1]:.2f})')
print('\n→ Bootstrap and t-interval give very similar results for the mean!')

### 11.2 Using scipy.stats.bootstrap (Modern Approach)

Since SciPy 1.7, there's a built-in `bootstrap()` function that handles all the details, including the BCa method.

Use `scipy.stats.bootstrap()` to compute bootstrap confidence intervals with the BCa method — the recommended approach in practice.

In [ ]:
# scipy.stats.bootstrap (available since SciPy 1.7)
from scipy.stats import bootstrap

# Data must be a sequence of 1-D arrays (tuple of arrays)
data = (original_sample,)

# Bootstrap CI for the mean (BCa method — best general method)
result_mean = bootstrap(data, statistic=np.mean, n_resamples=10_000, 
                        confidence_level=0.95, method='BCa', random_state=42)
print(f'BCa Bootstrap CI for mean:   ({result_mean.confidence_interval.low:.2f}, '
      f'{result_mean.confidence_interval.high:.2f})')
print(f'Bootstrap SE: {result_mean.standard_error:.2f}')

# Bootstrap CI for the median
result_median = bootstrap(data, statistic=np.median, n_resamples=10_000,
                          confidence_level=0.95, method='percentile', random_state=42)
print(f'\nPercentile Bootstrap CI for median: ({result_median.confidence_interval.low:.2f}, '
      f'{result_median.confidence_interval.high:.2f})')

# Bootstrap CI for trimmed mean (no formula exists!)
from scipy.stats import trim_mean
def trimmed_mean_10(x, axis):
    return trim_mean(x, proportiontocut=0.1, axis=axis)

result_trim = bootstrap(data, statistic=trimmed_mean_10, n_resamples=10_000,
                        confidence_level=0.95, method='percentile', random_state=42)
print(f'\nBootstrap CI for 10% trimmed mean: ({result_trim.confidence_interval.low:.2f}, '
      f'{result_trim.confidence_interval.high:.2f})')

### 11.3 Bootstrap for Complex Statistics

The bootstrap shines when we need confidence intervals for statistics where no formula exists.

Use bootstrap to estimate the CI for the Pearson correlation coefficient, the interquartile range, and the ratio of two means — statistics with no simple CI formulas.

In [ ]:
# Bootstrap for complex statistics
np.random.seed(42)

# Generate correlated data
n = 50
x = np.random.normal(100, 15, n)
y = 0.6 * x + np.random.normal(0, 10, n)  # Correlated with x

# Bootstrap CI for Pearson correlation
n_boot = 10_000
boot_corrs = []
for _ in range(n_boot):
    idx = np.random.choice(n, size=n, replace=True)
    boot_corrs.append(np.corrcoef(x[idx], y[idx])[0, 1])

corr_ci = (np.percentile(boot_corrs, 2.5), np.percentile(boot_corrs, 97.5))
print(f'Pearson r = {np.corrcoef(x, y)[0,1]:.4f}')
print(f'95% Bootstrap CI for r: ({corr_ci[0]:.4f}, {corr_ci[1]:.4f})')

# Bootstrap CI for IQR
boot_iqrs = []
for _ in range(n_boot):
    boot_s = np.random.choice(original_sample, size=len(original_sample), replace=True)
    boot_iqrs.append(np.percentile(boot_s, 75) - np.percentile(boot_s, 25))

iqr_ci = (np.percentile(boot_iqrs, 2.5), np.percentile(boot_iqrs, 97.5))
orig_iqr = np.percentile(original_sample, 75) - np.percentile(original_sample, 25)
print(f'\nSample IQR = {orig_iqr:.2f}')
print(f'95% Bootstrap CI for IQR: ({iqr_ci[0]:.2f}, {iqr_ci[1]:.2f})')

# Bootstrap CI for ratio of means (two groups)
group_a = np.array([45, 52, 48, 51, 47, 55, 50, 53, 49, 46])
group_b = np.array([38, 42, 40, 35, 44, 39, 41, 37, 43, 36])
ratio = group_a.mean() / group_b.mean()

boot_ratios = []
for _ in range(n_boot):
    ba = np.random.choice(group_a, size=len(group_a), replace=True)
    bb = np.random.choice(group_b, size=len(group_b), replace=True)
    boot_ratios.append(ba.mean() / bb.mean())

ratio_ci = (np.percentile(boot_ratios, 2.5), np.percentile(boot_ratios, 97.5))
print(f'\nMean ratio (A/B) = {ratio:.4f}')
print(f'95% Bootstrap CI for ratio: ({ratio_ci[0]:.4f}, {ratio_ci[1]:.4f})')

### 11.4 Reusable Bootstrap Function

Let's create a general-purpose bootstrap function that we can reuse throughout the course.

Build a reusable `bootstrap_analysis()` function that computes bootstrap CIs and produces diagnostic plots.

In [ ]:
def bootstrap_analysis(data, stat_func=np.mean, stat_name='Statistic',
                       n_boot=10_000, confidence=0.95, plot=True, random_state=42):
    """Complete bootstrap analysis with CI and visualization."""
    np.random.seed(random_state)
    n = len(data)
    
    # Compute observed statistic
    observed = stat_func(data)
    
    # Generate bootstrap distribution
    boot_stats = np.array([stat_func(np.random.choice(data, size=n, replace=True)) 
                           for _ in range(n_boot)])
    
    # Compute CI
    alpha = 1 - confidence
    ci_lower = np.percentile(boot_stats, 100 * alpha / 2)
    ci_upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    boot_se = np.std(boot_stats)
    
    # Print results
    print(f'=== Bootstrap Analysis: {stat_name} ===')
    print(f'Observed {stat_name}: {observed:.4f}')
    print(f'Bootstrap SE: {boot_se:.4f}')
    print(f'{confidence*100:.0f}% CI: ({ci_lower:.4f}, {ci_upper:.4f})')
    print(f'CI width: {ci_upper - ci_lower:.4f}')
    
    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        
        # Histogram of bootstrap distribution
        axes[0].hist(boot_stats, bins=50, color='steelblue', alpha=0.7, 
                     edgecolor='white', density=True)
        axes[0].axvline(observed, color='red', linestyle='--', linewidth=2,
                       label=f'Observed = {observed:.2f}')
        axes[0].axvline(ci_lower, color='orange', linestyle=':', linewidth=2,
                       label=f'{confidence*100:.0f}% CI')
        axes[0].axvline(ci_upper, color='orange', linestyle=':', linewidth=2)
        axes[0].fill_betweenx([0, axes[0].get_ylim()[1]*1.1], ci_lower, ci_upper, 
                              alpha=0.15, color='orange')
        axes[0].set_title(f'Bootstrap Distribution of {stat_name}', fontsize=13)
        axes[0].set_xlabel(stat_name, fontsize=12)
        axes[0].legend(fontsize=10)
        
        # QQ plot to check normality of bootstrap distribution
        stats.probplot(boot_stats, dist='norm', plot=axes[1])
        axes[1].set_title('Q-Q Plot of Bootstrap Distribution', fontsize=13)
        
        plt.tight_layout()
        plt.show()
    
    return {'observed': observed, 'se': boot_se, 'ci': (ci_lower, ci_upper), 
            'boot_stats': boot_stats}

# Demo: bootstrap analysis of median income
sample_incomes = np.random.lognormal(mean=10.5, sigma=0.5, size=80)
result = bootstrap_analysis(sample_incomes, stat_func=np.median, stat_name='Median Income')

---
## 12. Practical Example: Customer Satisfaction Survey

A company wants to estimate customer satisfaction from a survey. We'll apply everything we learned: sampling, CIs, and bootstrap.

### Step 1: Generate the Population and Sample

Create a realistic customer database and draw a stratified sample.

Build a simulated customer population with demographics and satisfaction scores, then draw a stratified sample by membership tier.

In [ ]:
np.random.seed(42)

# Simulate a population of 10,000 customers
N = 10_000
tiers = np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], size=N, 
                          p=[0.50, 0.30, 0.15, 0.05])

# Satisfaction scores differ by tier (higher tiers generally more satisfied)
tier_params = {'Bronze': (6.0, 2.0), 'Silver': (6.8, 1.8), 
               'Gold': (7.5, 1.5), 'Platinum': (8.2, 1.2)}

satisfaction = np.array([np.random.normal(*tier_params[t]) for t in tiers]).clip(1, 10)

customers = pd.DataFrame({
    'customer_id': range(1, N+1),
    'tier': tiers,
    'satisfaction': satisfaction.round(1),
    'tenure_years': np.random.exponential(3, N).clip(0.1, 20).round(1)
})

print(f'Population size: {len(customers):,}')
print(f'\nPopulation satisfaction by tier:')
print(customers.groupby('tier')['satisfaction'].agg(['mean', 'std', 'count']).round(2))
print(f'\nOverall population mean satisfaction: {customers["satisfaction"].mean():.2f}')

### Step 2: Stratified Sampling by Tier

Draw a proportional stratified sample of 200 customers to ensure each membership tier is represented.

Perform stratified sampling to draw 200 customers proportionally across tiers and compare with the population.

In [ ]:
# Stratified sample: proportional by tier
sample_size = 200
survey_sample = customers.groupby('tier', group_keys=False).apply(
    lambda x: x.sample(n=max(1, round(sample_size * len(x) / N)), random_state=42)
)

print(f'Survey sample size: {len(survey_sample)}')
print(f'\nSample satisfaction by tier:')
print(survey_sample.groupby('tier')['satisfaction'].agg(['mean', 'std', 'count']).round(2))
print(f'\nSample mean satisfaction: {survey_sample["satisfaction"].mean():.2f}')
print(f'Population mean satisfaction: {customers["satisfaction"].mean():.2f}')

### Step 3: Confidence Intervals

Compute various confidence intervals for the overall satisfaction mean and for the proportion of "satisfied" customers (score >= 7).

Calculate t-interval for the mean and Wilson CI for the proportion, along with bootstrap CIs for both statistics.

In [ ]:
sat_scores = survey_sample['satisfaction'].values
n = len(sat_scores)

# --- CI for mean satisfaction ---
x_bar = sat_scores.mean()
se = sat_scores.std(ddof=1) / np.sqrt(n)
t_ci = stats.t.interval(0.95, df=n-1, loc=x_bar, scale=se)

print('=== 95% CI for Mean Satisfaction ===')
print(f'T-interval: ({t_ci[0]:.3f}, {t_ci[1]:.3f})')

# Bootstrap CI for mean
boot_means_ci = [np.random.choice(sat_scores, size=n, replace=True).mean() 
                 for _ in range(10_000)]
boot_ci = (np.percentile(boot_means_ci, 2.5), np.percentile(boot_means_ci, 97.5))
print(f'Bootstrap CI: ({boot_ci[0]:.3f}, {boot_ci[1]:.3f})')

# --- CI for proportion satisfied (score >= 7) ---
satisfied = (sat_scores >= 7).sum()
p_hat = satisfied / n

wilson_ci = proportion_confint(satisfied, n, alpha=0.05, method='wilson')
print(f'\n=== 95% CI for Proportion Satisfied (score >= 7) ===')
print(f'p̂ = {p_hat:.3f} ({satisfied}/{n})')
print(f'Wilson CI: ({wilson_ci[0]:.3f}, {wilson_ci[1]:.3f})')

# --- CI by tier ---
print('\n=== 95% T-interval by Tier ===')
for tier in ['Bronze', 'Silver', 'Gold', 'Platinum']:
    tier_data = survey_sample[survey_sample['tier'] == tier]['satisfaction']
    ci = stats.t.interval(0.95, df=len(tier_data)-1, loc=tier_data.mean(),
                          scale=tier_data.std(ddof=1)/np.sqrt(len(tier_data)))
    print(f'  {tier:>10}: ({ci[0]:.2f}, {ci[1]:.2f})  [n={len(tier_data)}]')

### Step 4: Visualize the Results

Create a comprehensive visualization of our estimation results.

Plot the sample distribution, bootstrap distribution, and confidence intervals for each tier in a multi-panel figure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Sample distribution vs population
ax = axes[0, 0]
ax.hist(customers['satisfaction'], bins=40, density=True, alpha=0.4, color='gray', label='Population')
ax.hist(sat_scores, bins=20, density=True, alpha=0.6, color='steelblue', label='Sample')
ax.axvline(customers['satisfaction'].mean(), color='red', linestyle='--', label=f'Pop μ = {customers["satisfaction"].mean():.2f}')
ax.axvline(x_bar, color='blue', linestyle='--', label=f'Sample x̄ = {x_bar:.2f}')
ax.set_title('Sample vs Population Distribution', fontsize=13)
ax.set_xlabel('Satisfaction Score', fontsize=11)
ax.legend(fontsize=9)

# 2. Bootstrap distribution with CI
ax = axes[0, 1]
ax.hist(boot_means_ci, bins=50, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(boot_ci[0], color='orange', linestyle='--', linewidth=2, label='95% CI')
ax.axvline(boot_ci[1], color='orange', linestyle='--', linewidth=2)
ax.axvline(x_bar, color='red', linestyle='-', linewidth=2, label=f'x̄ = {x_bar:.2f}')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1], 
                 boot_ci[0], boot_ci[1], alpha=0.2, color='orange')
ax.set_title('Bootstrap Distribution of Mean', fontsize=13)
ax.set_xlabel('Sample Mean', fontsize=11)
ax.legend(fontsize=10)

# 3. CI by tier (forest plot)
ax = axes[1, 0]
tiers_list = ['Bronze', 'Silver', 'Gold', 'Platinum']
colors_tier = ['#cd7f32', 'silver', 'gold', '#e5e4e2']
for i, tier in enumerate(tiers_list):
    tier_data = survey_sample[survey_sample['tier'] == tier]['satisfaction']
    ci = stats.t.interval(0.95, df=len(tier_data)-1, loc=tier_data.mean(),
                          scale=tier_data.std(ddof=1)/np.sqrt(len(tier_data)))
    pop_mean_tier = customers[customers['tier'] == tier]['satisfaction'].mean()
    
    ax.plot([ci[0], ci[1]], [i, i], linewidth=3, color=colors_tier[i], zorder=2)
    ax.plot(tier_data.mean(), i, 'ko', markersize=8, zorder=3)
    ax.plot(pop_mean_tier, i, 'r^', markersize=10, zorder=3)

ax.set_yticks(range(len(tiers_list)))
ax.set_yticklabels(tiers_list, fontsize=11)
ax.set_xlabel('Satisfaction Score', fontsize=11)
ax.set_title('95% CI by Tier (● sample mean, ▲ pop mean)', fontsize=13)

# 4. Sample size needed for different margins of error
ax = axes[1, 1]
E_vals = np.linspace(0.1, 1.0, 50)
s_est = sat_scores.std(ddof=1)
n_needed = [(stats.norm.ppf(0.975) * s_est / E) ** 2 for E in E_vals]
ax.plot(E_vals, n_needed, 'b-', linewidth=2)
ax.axhline(200, color='red', linestyle='--', alpha=0.5, label=f'Current n=200')
ax.set_xlabel('Desired Margin of Error', fontsize=11)
ax.set_ylabel('Required Sample Size', fontsize=11)
ax.set_title('Sample Size Planning', fontsize=13)
ax.legend(fontsize=10)

fig.suptitle('Customer Satisfaction Survey: Complete Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Step 5: Summary Report

Generate a professional summary of our findings.

Compile all estimation results into a structured summary report.

In [ ]:
print('=' * 60)
print('    CUSTOMER SATISFACTION SURVEY — ESTIMATION REPORT')
print('=' * 60)
print(f'\nSampling Method: Stratified by membership tier')
print(f'Sample Size: n = {n}')
print(f'Population Size: N = {N:,}')
print(f'Sampling Fraction: {n/N:.1%}')

print(f'\n--- Overall Satisfaction ---')
print(f'Point estimate (mean): {x_bar:.2f}')
print(f'95% T-interval: ({t_ci[0]:.2f}, {t_ci[1]:.2f})')
print(f'95% Bootstrap CI: ({boot_ci[0]:.2f}, {boot_ci[1]:.2f})')
print(f'Margin of error: ±{(t_ci[1]-t_ci[0])/2:.2f}')

print(f'\n--- Proportion Satisfied (score ≥ 7) ---')
print(f'Point estimate: {p_hat:.1%}')
print(f'95% Wilson CI: ({wilson_ci[0]:.1%}, {wilson_ci[1]:.1%})')

print(f'\n--- Sample Size Recommendations ---')
for target_E in [0.2, 0.3, 0.5]:
    n_rec = int(np.ceil((stats.norm.ppf(0.975) * s_est / target_E) ** 2))
    print(f'  For ±{target_E:.1f} margin: n = {n_rec}')

print(f'\n--- Verification ---')
true_mean = customers['satisfaction'].mean()
print(f'True population mean: {true_mean:.2f}')
print(f'Does 95% CI contain true mean? {t_ci[0] < true_mean < t_ci[1]}')
print('=' * 60)

---
## 13. Summary

| Section | Key Concepts | Python Functions |
|---------|-------------|------------------|
| 1. Population vs Sample | Parameters (μ, σ) vs statistics (x̄, s) | `np.mean()`, `np.std(ddof=1)` |
| 2. Sampling Methods | SRS, stratified, systematic, cluster | `df.sample()`, `np.random.choice()` |
| 3. Sampling Bias | Selection, survivorship, response bias | — |
| 4. Sampling Distribution | SE = σ/√n, distribution of statistics | Simulation with list comprehension |
| 5. Central Limit Theorem | x̄ → Normal as n increases, any population | Simulation + visualization |
| 6. Point Estimation | Unbiased, consistent, efficient estimators | `ddof=1` for unbiased variance |
| 7. CI for Mean | z-interval (σ known), t-interval (σ unknown) | `stats.t.interval()` |
| 8. CI for Proportion | Wald (simple), Wilson (recommended) | `proportion_confint()` |
| 9. Sample Size | n = (z*σ/E)², n = z²p(1-p)/E² | Manual formula |
| 10. Bootstrap | Resample with replacement, any statistic | `np.random.choice(replace=True)` |
| 11. Bootstrap CI | Percentile, BCa methods | `scipy.stats.bootstrap()` |
| 12. Practical Example | Complete survey estimation pipeline | All above combined |

### Key Takeaways

1. **The sampling method matters** — biased sampling cannot be fixed by larger samples
2. **The CLT is why statistics works** — sample means are approximately normal regardless of population shape
3. **Confidence intervals quantify uncertainty** — always report a CI, not just a point estimate
4. **Bootstrap is incredibly versatile** — use it whenever formulas don't exist or assumptions are questionable
5. **Sample size planning saves resources** — calculate n before collecting data

---
## 14. Homework

### Task 1: CLT Exploration
Generate a population from a **Beta(2, 5)** distribution (use `np.random.beta(2, 5, 100000)`). 
1. Plot the population distribution
2. Draw 3000 samples of sizes n = 5, 15, 30, and 100
3. Plot the sampling distribution of the mean for each n
4. Compute the theoretical SE vs observed SE for each n
5. At what sample size does the sampling distribution look approximately normal?

### Task 2: Confidence Interval Comparison
A factory produces bolts with a target diameter of 10mm. A sample of 25 bolts gives:
`bolts = [10.1, 9.8, 10.3, 9.9, 10.0, 10.2, 9.7, 10.1, 10.0, 9.9, 10.4, 9.8, 10.1, 10.0, 9.9, 10.2, 10.3, 9.8, 10.1, 10.0, 9.7, 10.2, 10.0, 10.1, 9.9]`
1. Compute the 95% t-interval for the mean diameter
2. Compute the 95% bootstrap percentile CI (10,000 resamples)
3. Compute the 99% t-interval — how much wider is it than the 95%?
4. Does the target of 10mm fall within your 95% CI?

### Task 3: Bootstrap for Non-Standard Statistics
Using the bolt data from Task 2:
1. Compute a 95% bootstrap CI for the **standard deviation**
2. Compute a 95% bootstrap CI for the **coefficient of variation** (CV = s/x̄)
3. Compute a 95% bootstrap CI for the **5th percentile** (quality control: what's the smallest bolt we can expect?)
4. Visualize the bootstrap distribution for each statistic

### Task 4: Survey Sample Size Planning
You are planning a survey to estimate:
- (a) Average monthly spending among college students (you estimate σ ≈ $150)
- (b) The proportion of students who use public transportation

For each:
1. How many students do you need to survey for a 95% CI with margin of error ±$20 (for spending) or ±4% (for proportion)?
2. How does the required n change if you want 99% confidence instead?
3. For the proportion, compare the required n when your best guess is p=0.3 vs p=0.5
4. Create a visualization showing required n vs margin of error for both scenarios

---
### Next Week Preview

**Week 8: Hypothesis Testing** — Can we go beyond estimation and make *decisions* about population parameters? We'll learn about null and alternative hypotheses, p-values, z-tests, t-tests, chi-square tests, and the critical distinction between Type I and Type II errors. We'll also explore effect sizes — because statistical significance is not the same as practical significance.

---
*Applied Statistics with Python (2026) | Week 7 | Sampling & Estimation*